# Site 54 across bnAb backgrounds

Distribution of the site 54 effect across every measured background, split by antigen, one column per antibody that varies Kabat 54. Effects are matched pairs of measured genotypes, mutant minus germline.

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Config

In [ ]:
data_dir = "../data"
fig_dir = "../figures"

target_kabat = 54
numbering = "kabat"
antigens = ["CH505TF", "SF162"]
antigen_hex = {"CH505TF": "#E66100", "SF162": "#5D3A9B"}
site_cols = [f"site_{i}" for i in range(10)]
floor = 6.0

# column order of the published panel, antibodies without a Kabat 54 site drop out on their own
bnab_order = ["1NC9", "VRC01", "NIH45-46", "N49P7", "VRC07", "VRC-PG20"]
all_bnabs = ["1NC9", "CH235.12", "N49P7", "NIH45-46", "VRC-PG20", "VRC01", "VRC07", "VRC13", "VRC16"]

# display name to file stem; the hyphens and the dot do not appear in any filename
name_to_stem = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}

violin_width = 0.85
quartile_lw = 1.0
y_limits = None        # set a tuple to clip the N49P7 outlier tail
fig_size = (9.0, 5.2)
png_dpi = 300

plt.rcParams["svg.fonttype"] = "path"   # illustrator drops live text, see the project export note
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

os.makedirs(fig_dir, exist_ok=True)

## Loading

In [ ]:
def site_54(bnab):
    """Site index and residues at Kabat 54, or None when this library does not vary it."""
    meta = pd.read_csv(f"{data_dir}/{name_to_stem[bnab]}_meta.csv")
    pos_col = "kabat_position" if numbering == "kabat" else "imgt_position"
    for r in meta.itertuples():
        if str(getattr(r, pos_col)).strip() == str(target_kabat):
            return int(r.site.split("_")[1]), r.germline_aa, r.mature_aa
    return None


def load_genotypes(bnab):
    """Collapse replicates to one measured affinity per genotype and antigen."""
    kds = pd.read_csv(f"{data_dir}/{name_to_stem[bnab]}_kds.csv")
    kds["tok"] = kds[site_cols].agg("".join, axis=1)
    return kds.groupby(["tok", "antigen"], as_index=False)["nlog10_Kd"].mean()


def matched_deltas(bnab, antigen, site_idx):
    """Mutant minus germline at site 54 over every background measured in both states.

    Pairs with both members on the assay floor contribute exactly zero, so mass at zero
    is censoring rather than a null effect.
    """
    df = load_genotypes(bnab)
    sub = df[df.antigen == antigen]
    vals = dict(zip(sub["tok"], sub["nlog10_Kd"]))
    deltas = []
    for tok, germline in vals.items():
        if tok[site_idx] != "G":
            continue
        partner = tok[:site_idx] + "M" + tok[site_idx + 1:]
        if partner in vals:
            deltas.append(vals[partner] - germline)
    return np.asarray(deltas, float)


def save_figure(fig, stem):
    """Write png, pdf and svg, then check the svg has no live text."""
    path = os.path.join(fig_dir, stem)
    fig.savefig(f"{path}.png", dpi=png_dpi, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    fig.savefig(f"{path}.svg", bbox_inches="tight")
    assert open(f"{path}.svg").read().count("<text") == 0, "live text in svg, check svg.fonttype"
    print("wrote", stem, "png pdf svg")

## Figure S12

In [ ]:
def half_violin(ax, vals, pos, side, color):
    """One side of a split violin, with median and quartile rules drawn across that half."""
    body = ax.violinplot([vals], positions=[pos], widths=violin_width,
                         showextrema=False, showmedians=False)["bodies"][0]
    verts = body.get_paths()[0].vertices
    lo, hi = (-np.inf, pos) if side == "left" else (pos, np.inf)
    verts[:, 0] = np.clip(verts[:, 0], lo, hi)
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_linewidth(0.6)
    body.set_alpha(0.9)

    reach = np.abs(verts[:, 0] - pos).max()
    for q in np.percentile(vals, [25, 50, 75]):
        x0, x1 = (pos - reach, pos) if side == "left" else (pos, pos + reach)
        ax.plot([x0, x1], [q, q], color="black", lw=quartile_lw, zorder=3)


def build_site54_violins(out_stem=None):
    """Site 54 effect across backgrounds, CH505TF on the left half and SF162 on the right."""
    fig, ax = plt.subplots(figsize=fig_size)
    labels, positions = [], []
    for i, bnab in enumerate(bnab_order):
        resolved = site_54(bnab)
        if resolved is None:
            print("no kabat 54 site, skipping", bnab)
            continue
        site_idx, germline_aa, mature_aa = resolved
        for antigen, side in zip(antigens, ["left", "right"]):
            half_violin(ax, matched_deltas(bnab, antigen, site_idx), i, side, antigen_hex[antigen])
        labels.append(f"{bnab}\n{germline_aa}{target_kabat}{mature_aa}")
        positions.append(i)

    ax.axhline(0, ls="--", lw=1, color="#999999", zorder=1)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_xlim(-0.6, len(bnab_order) - 0.4)
    if y_limits:
        ax.set_ylim(y_limits)
    ax.set_ylabel(r"$\Delta$ $-\log_{10}\mathrm{K_D}$  (mutant $-$ germline)", fontsize=11)
    ax.set_title("Effect of site 54 mutation across backgrounds", fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3)

    keys = [plt.Rectangle((0, 0), 1, 1, facecolor=antigen_hex[a], edgecolor="black", lw=0.6, label=a)
            for a in antigens]
    ax.legend(handles=keys, title="antigen", frameon=False, loc="upper right",
              fontsize=9, title_fontsize=9)

    fig.tight_layout()
    if out_stem:
        save_figure(fig, out_stem)
    return fig

In [ ]:
build_site54_violins("fig_s12_site54_effect_across_backgrounds")
plt.show()

### Numbers for the caption

In [ ]:
rows = []
for bnab in bnab_order:
    resolved = site_54(bnab)
    if resolved is None:
        continue
    site_idx, germline_aa, mature_aa = resolved
    for antigen in antigens:
        d = matched_deltas(bnab, antigen, site_idx)
        rows.append({"bnab": bnab, "sub": f"{germline_aa}{target_kabat}{mature_aa}",
                     "antigen": antigen, "n_pairs": len(d),
                     "q25": np.percentile(d, 25), "median": np.median(d),
                     "q75": np.percentile(d, 75), "frac_positive": (d > 0).mean()})
display(pd.DataFrame(rows).round(3))

## Walsh-Hadamard variance footprint

Exploratory, not in the paper. How much of the variance at each interaction order
involves site 54, computed on the fitted landscape rather than on measured pairs.
Only the six complexes with usable fits should be read here.

In [ ]:
def params_path(bnab, antigen):
    """Best-fit coefficients for one complex, or None when the fit was not kept."""
    path = f"{data_dir}/{name_to_stem[bnab]}_{antigen}_BEST_df_params.csv"
    return path if os.path.exists(path) else None


def load_terms(path):
    """Coefficient table as (site tuple, value) pairs, with the intercept as an empty tuple."""
    table = pd.read_csv(path)
    terms = []
    for r in table.itertuples():
        t = str(r.term).strip()
        sites = tuple(sorted(int(p.split("_")[1]) for p in t.split(","))) if t.startswith("site_") else ()
        terms.append((sites, float(r.coefficient)))
    return terms


def predict_latent(terms, n_sites=10):
    """Latent phenotype over all 2^n genotypes, integer index read as a bitmask of mutant sites."""
    n = 1 << n_sites
    idx = np.arange(n)
    y = np.zeros(n)
    for sites, c in terms:
        if not sites:
            y += c
            continue
        mask = 0
        for s in sites:
            mask |= 1 << s
        y += c * ((idx & mask) == mask)
    return y


def fwht(a):
    """In-place fast Walsh-Hadamard transform, unnormalized."""
    a = np.asarray(a, float).copy()
    n, h = len(a), 1
    while h < n:
        for i in range(0, n, h * 2):
            x, y = a[i:i + h].copy(), a[i + h:i + 2 * h].copy()
            a[i:i + h] = x + y
            a[i + h:i + 2 * h] = x - y
        h *= 2
    return a


def site54_variance(y, site_idx, n_sites=10):
    """Per order, the variance carried by Walsh functions that touch site 54."""
    n = 1 << n_sites
    sq = (fwht(y) / n) ** 2
    sq[0] = 0.0                                   # order 0 is the mean, not variance
    order = np.array([bin(s).count("1") for s in range(n)])
    touches = ((np.arange(n) >> site_idx) & 1).astype(bool)

    foot, share = {}, {}
    for k in range(1, n_sites + 1):
        at_k = order == k
        total_k = sq[at_k].sum()
        foot[k] = sq[at_k & touches].sum()
        share[k] = foot[k] / total_k if total_k > 0 else 0.0
    return foot, share


def variance_table():
    """One row per complex with the site 54 footprint by order and its share within each order."""
    rows = []
    for bnab in all_bnabs:
        resolved = site_54(bnab)
        if resolved is None:
            continue
        site_idx, germline_aa, mature_aa = resolved
        for antigen in antigens:
            path = params_path(bnab, antigen)
            if path is None:
                print("no best fit params for", bnab, antigen)
                continue
            foot, share = site54_variance(predict_latent(load_terms(path)), site_idx)
            total = sum(foot.values())
            row = {"bnab": bnab, "sub": f"{germline_aa}{target_kabat}{mature_aa}", "antigen": antigen,
                   "epistatic_frac": 1 - foot[1] / total if total > 0 else np.nan}
            row.update({f"foot_{k}": foot[k] / total if total > 0 else np.nan for k in range(1, 6)})
            row.update({f"share_{k}": share[k] for k in range(1, 5)})
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
variance = variance_table()
display(variance.round(3))

In [ ]:
def build_variance_bars(variance, antigen, out_stem=None):
    """Share of each order's variance that involves site 54, one group per complex."""
    sub = variance[variance.antigen == antigen]
    orders = [1, 2, 3, 4]
    order_hex = {1: "#4a6fa5", 2: "#6a8fc5", 3: "#d98b5f", 4: "#c56b3f"}
    x = np.arange(len(sub))
    w = 0.8 / len(orders)

    fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(sub)), 4.6))
    for j, k in enumerate(orders):
        ax.bar(x + j * w - 0.4 + w / 2, 100 * sub[f"share_{k}"], width=w,
               color=order_hex[k], label=f"order {k}")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r.bnab}\n({r.sub})" for r in sub.itertuples()], fontsize=9)
    ax.set_ylabel("percent of order variance involving site 54")
    ax.set_title(f"site 54 share within each interaction order: {antigen}", fontsize=12)
    ax.legend(frameon=False, fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    if out_stem:
        save_figure(fig, out_stem)
    return fig


for antigen in antigens:
    build_variance_bars(variance, antigen, f"site54_order_share_{antigen}")
plt.show()